> ⚠️ **Before you start — attach the cluster**
>
> This lab requires a **classic compute cluster with Photon disabled**. Serverless compute does not support the Spark configuration options used in the exercises, and Photon automatically optimises away the skew and shuffle problems this lab is designed to expose — you will not see anything abnormal in the Spark UI if either is active.
>
> 1. In the compute selector at the top of this notebook, click **Connect**.
> 2. Select the **`perf-lab`** cluster (created in the lab setup instructions — non-Photon runtime, `spark.databricks.photon.enabled false` set in Spark config).
> 3. Wait until the cluster status shows **Running** before running any cells.

## Exercise 1: Set up your environment

Before investigating performance problems you need data to work with. This exercise creates a Unity Catalog structure and generates two Delta tables:

- **`perf_lab.data.transactions`** — 500 000 rows with an intentionally skewed `customer_id` column (~70 % of rows belong to a single customer)
- **`perf_lab.data.customers`** — 200 rows (a small dimension table)

Run all cells in this exercise from top to bottom before continuing.

### Task 1.1: Create the Unity Catalog structure

This cell is provided — run it to create the catalog and schema.

In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
# Use the current catalog to reliably find the workspace default catalog,
# regardless of its naming convention.
default_catalog = spark.catalog.currentCatalog()

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

In [ ]:
# ✅ Run this cell — creates the catalog and schema
spark.sql(f"CREATE CATALOG IF NOT EXISTS perf_lab MANAGED LOCATION '{storage_root}' COMMENT 'Performance troubleshooting lab'")
spark.sql("CREATE SCHEMA IF NOT EXISTS perf_lab.data COMMENT 'Transaction data for performance exercises'")
print("✅ Catalog and schema ready.")

### Task 1.2: Generate the transactions table

The dataset is generated entirely in Spark — no external files are needed.

**Intentional skew:** `rand(seed=42) < 0.70` routes approximately 70 % of all rows to `customer_001`. Every other row is distributed across `customer_002` through `customer_200`. You will see the consequences of this imbalance in Exercise 2.

In [ ]:
# ✅ Run this cell — generates 500 000 transactions with a heavily skewed customer_id
from pyspark.sql import functions as F

n_rows = 500_000

df_transactions = (
    spark.range(n_rows)
    .withColumn(
        "customer_id",
        F.when(F.rand(seed=42) < 0.70, F.lit("customer_001"))
         .otherwise(
             F.concat(
                 F.lit("customer_"),
                 F.lpad((F.floor(F.rand(seed=99) * 199) + 2).cast("string"), 3, "0")
             )
         )
    )
    .withColumn("amount",   F.round(F.rand(seed=7) * 999 + 1, 2))
    .withColumn("txn_type", F.when(F.rand(seed=3) < 0.5, F.lit("purchase")).otherwise(F.lit("refund")))
    .withColumn("txn_id",   F.concat(F.lit("txn_"), F.col("id").cast("string")))
    .drop("id")
)

df_transactions.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.transactions")
count = spark.table("perf_lab.data.transactions").count()
print(f"✅ Written {count:,} rows to perf_lab.data.transactions")

### Task 1.3: Generate the customers table

This small dimension table has 200 rows — one row per customer. It is tiny enough to fit comfortably in memory on a single executor. Keep that fact in mind; it will be important in Exercise 3.

In [ ]:
# ✅ Run this cell — generates 200 customer dimension rows
customer_rows = [
    (f"customer_{i:03d}", f"Customer {i}", f"region_{(i % 5) + 1}")
    for i in range(1, 201)
]

df_customers = spark.createDataFrame(customer_rows, ["customer_id", "name", "region"])
df_customers.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.customers")
print(f"✅ Written {df_customers.count()} rows to perf_lab.data.customers")

### Task 1.4: Verify the skew

Run the cell below to confirm that the data is heavily imbalanced. The output shows the top customers by transaction count and their share of the total dataset.

In [ ]:
# ✅ Run this cell — shows the distribution of rows across customer_id values
df = spark.table("perf_lab.data.transactions")
total = df.count()

top_customers = (
    df.groupBy("customer_id")
      .agg(F.count("*").alias("txn_count"))
      .withColumn("pct_of_total", F.round(F.col("txn_count") / total * 100, 1))
      .orderBy(F.desc("txn_count"))
      .limit(5)
)

print(f"Total rows: {total:,}")
print("Top 5 customers by transaction count:")
top_customers.show()

---

## Exercise 2: Expose data skew

To make skew clearly visible in the Spark UI, you need to turn off **Adaptive Query Execution (AQE)**. With AQE enabled, Databricks automatically splits oversized partitions at runtime, which masks the skew but also hides the problem in the UI.

You will run two operations that both shuffle on `customer_id`:
1. A **groupBy aggregation** — the shuffle routes ~70 % of rows to a single reducer task
2. A **sort-merge join** between transactions and customers — auto-broadcast is disabled so Spark must shuffle both sides

After running the cells below, **return to the lab instructions page** and follow the steps in **Part 2** to investigate the results in the Spark UI.

### Task 2.1: Disable AQE and set shuffle partitions

In [ ]:
# ✅ Run this cell — disables AQE and auto-broadcast so skew is fully visible in the Spark UI
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "false")
spark.conf.set("spark.sql.adaptive.enabled",                 "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",       "-1")    # Force sort-merge join
spark.conf.set("spark.sql.shuffle.partitions",               "50")

print("AQE disabled | auto-broadcast disabled | shuffle partitions: 50")

### Task 2.2: Run a skewed aggregation

The `groupBy("customer_id")` triggers a shuffle. Because `customer_001` has ~350 000 rows, one partition (and therefore one task) receives roughly 70 % of all data. All other partitions share the remaining 30 %.

In [ ]:
# ✅ Run this cell — triggers a skewed shuffle on customer_id
import time

df_tx = spark.table("perf_lab.data.transactions")

start = time.time()
result_agg = (
    df_tx
    .groupBy("customer_id", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count"),
        F.avg("amount").alias("avg_amount")
    )
)
result_agg.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.agg_skewed")
elapsed = time.time() - start
print(f"✅ Aggregation complete in {elapsed:.1f}s — {spark.table('perf_lab.data.agg_skewed').count()} rows")

### Task 2.3: Run a skewed sort-merge join

Auto-broadcast is disabled (`autoBroadcastJoinThreshold = -1`), so Spark must shuffle **both** sides of the join and use a sort-merge strategy. The skewed `customer_id` distribution means one reducer task receives a disproportionately large partition from the transactions side.

In [ ]:
# ✅ Run this cell — triggers a skewed sort-merge join
df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()
result_join = (
    df_tx
    .join(df_cust, "customer_id", "left")
    .groupBy("region", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count")
    )
)
result_join.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.join_skewed")
elapsed = time.time() - start
print(f"✅ Sort-merge join complete in {elapsed:.1f}s — {spark.table('perf_lab.data.join_skewed').count()} rows")

> **Now return to the lab instructions page and follow the steps in Part 2 to investigate the Spark UI.**
>
> When you have finished your investigation, continue with Exercise 3 below.

---

## Exercise 3: Fix the data skew

You have two complementary tools for addressing skew:

| Technique | How it works | Best when |
|-----------|-------------|----------|
| **Broadcast join** | Sends the small table to every executor — no shuffle of either side | One side of a join is small |
| **AQE skew join** | Splits oversized shuffle partitions dynamically at runtime | General-purpose skew in joins and aggregations |

In this exercise you will apply both.

### 🎯 Task 3.1: Fix the join using a broadcast hint

The `customers` table has only **200 rows**. It easily fits in memory on every executor. Broadcasting it eliminates the shuffle of the `customers` side entirely and means the `customer_id` skew on the transactions side no longer causes any task imbalance.

**Your task:** Rewrite the join from Task 2.3 to use a broadcast hint for the `customers` DataFrame.

💡 **Hints:**
- PySpark syntax: wrap the small DataFrame with `F.broadcast(df)`
- SQL hint syntax: `/*+ BROADCAST(alias) */` inside a SQL string
- The broadcast hint tells Spark to skip the sort-merge strategy and instead copy the small table to every worker

🤖 **Genie Code prompt:**
> *"How do I use a broadcast join hint in PySpark to avoid shuffling a small dimension table?"*

In [ ]:
# 🎯 YOUR TASK: Rewrite this join to use a broadcast hint for df_cust
# Measure the elapsed time and compare it with the sort-merge join in Task 2.3
import time
from pyspark.sql import functions as F

df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()

# TODO: replace the plain join below with a broadcast join
result_broadcast = (
    df_tx
    .join(df_cust, "customer_id", "left")   # ← apply the broadcast hint here
    .groupBy("region", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count")
    )
)
result_broadcast.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.join_broadcast")
elapsed = time.time() - start
print(f"Join complete in {elapsed:.1f}s — {spark.table('perf_lab.data.join_broadcast').count()} rows")

### Task 3.2: Re-enable AQE and observe automatic skew handling

With `spark.sql.adaptive.skewJoin.enabled = true`, AQE automatically detects skewed partitions during a shuffle and splits them into smaller sub-tasks. This handles skew without any code changes — useful when you cannot (or do not want to) broadcast a table.

The cell below re-enables AQE and re-runs the original sort-merge join from Task 2.3 (no broadcast hint). Compare the execution time and check the Spark UI to see how AQE splits the skewed partition.

In [ ]:
# ✅ Run this cell — re-enables AQE and re-runs the sort-merge join
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled",                 "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled",        "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",       "10485760")  # Reset to 10 MB

print("AQE enabled | skewJoin enabled | auto-broadcast threshold: 10 MB")

df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()
result_aqe = (
    df_tx
    .join(df_cust, "customer_id", "left")
    .groupBy("region", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count")
    )
)
result_aqe.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.join_aqe")
elapsed = time.time() - start
print(f"✅ AQE join complete in {elapsed:.1f}s — {spark.table('perf_lab.data.join_aqe').count()} rows")
print("\nCheck the Spark UI — AQE splits the skewed partition into multiple sub-tasks.")

---

## Exercise 4: Expose excessive shuffle

Shuffle is unavoidable when Spark needs to redistribute data — for example, for a `groupBy` or a `join`. However, it is easy to introduce *unnecessary* shuffles by calling `repartition()` before or after operations that already involve a shuffle.

The pipeline in this exercise performs a `groupBy` followed by a `join` and a `sort`, but adds **three extra `repartition()` calls** that each trigger a full shuffle for no benefit. The Spark UI will show multiple **Exchange** nodes in the DAG — more than the computation actually requires.

Run the cell below, then **return to the lab instructions page** and follow the steps in **Part 4** to investigate the shuffle metrics.

### Task 4.1: Reset configs and run the shuffle-heavy pipeline

In [ ]:
# ✅ Run this cell — disables AQE and auto-broadcast so all shuffles are visible
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "false")
spark.conf.set("spark.sql.adaptive.enabled",                 "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",       "-1")
spark.conf.set("spark.sql.shuffle.partitions",               "200")

print("AQE disabled | auto-broadcast disabled | shuffle partitions: 200")

In [ ]:
# ✅ Run this cell — shuffle-heavy pipeline with unnecessary repartitions
#
# Shuffle 1: repartition(100, "txn_type")     — upfront repartition before groupBy (not needed)
# Shuffle 2: groupBy aggregation              — necessary shuffle on customer_id
# Shuffle 3: repartition(200, "customer_id")  — re-shuffle before join (not needed)
# Shuffle 4: sort-merge join with customers   — shuffle because broadcast is disabled
# Shuffle 5: repartition(50)                  — post-join repartition (not needed)
# Shuffle 6: orderBy                          — necessary global sort

df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()
result_inefficient = (
    df_tx
    .repartition(100, "txn_type")                       # Shuffle 1: unnecessary
    .groupBy("customer_id", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count"),
        F.avg("amount").alias("avg_amount")
    )
    .repartition(200, "customer_id")                    # Shuffle 3: unnecessary
    .join(df_cust, "customer_id", "left")               # Shuffle 4: sort-merge (broadcast disabled)
    .repartition(50)                                    # Shuffle 5: unnecessary
    .orderBy("region", "txn_type")                      # Shuffle 6: necessary
)
result_inefficient.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.results_inefficient")
elapsed = time.time() - start
print(f"✅ Inefficient pipeline complete in {elapsed:.1f}s")
print(f"   Rows written: {spark.table('perf_lab.data.results_inefficient').count():,}")
print("\nNow return to the lab instructions page and investigate the Spark UI for Part 4.")

> **Return to the lab instructions page and follow the steps in Part 4 to investigate the shuffle metrics.**
>
> When you have finished your investigation, continue with Exercise 5 below.

---

## Exercise 5: Reduce shuffle overhead

The pipeline in Exercise 4 performs a `groupBy → join → sort`. That computation fundamentally requires **two** shuffles: one for the aggregation and one for the global sort. Everything else is overhead.

### 🎯 Challenge

Rewrite the pipeline to minimise the number of shuffles. Your optimised version should:

1. **Remove all unnecessary `repartition()` calls** — a `groupBy`, `join`, or `orderBy` already moves data; pre-shuffling does not help and only adds extra network I/O.
2. **Use a broadcast join** for the `customers` table to eliminate the sort-merge shuffle entirely.
3. **Re-enable AQE** before running so Spark can further optimise partition sizes dynamically.
4. **Keep the `orderBy`** — the output must still be sorted.

Measure the elapsed time and compare it with the Exercise 4 runtime.

💡 **Hints:**
- Every call to `repartition()` triggers a full shuffle — only keep it when you have a specific reason (e.g., controlling the number of output files written to storage).
- `customers` has 200 rows. Use `F.broadcast(df_cust)` to copy it to every executor.
- After the broadcast join, only the `orderBy` shuffle remains — that is the minimum for a sorted result.

🤖 **Genie Code prompt:**
> *"I have a PySpark pipeline with multiple repartition() calls before a groupBy and a join. Help me identify which repartitions are unnecessary and rewrite the pipeline to minimise the number of shuffles. Also apply a broadcast join for a small dimension table."*

In [ ]:
# 🎯 YOUR TASK: Rewrite the pipeline from Exercise 4 to minimise shuffle
#
# Re-enable AQE and auto-broadcast, then write an optimised version of the pipeline.
# Time it and compare with Exercise 4.

# Step 1: Re-enable AQE and broadcast threshold
# TODO: set spark.databricks.optimizer.adaptive.enabled and spark.sql.adaptive.enabled to true
# TODO: reset spark.sql.autoBroadcastJoinThreshold to 10485760 (10 MB)

df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()

# Step 2: Write the optimised pipeline
# TODO: remove unnecessary repartitions
# TODO: apply F.broadcast() to the customers join
result_optimised = (
    df_tx
    # ... your code here ...
)
result_optimised.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.results_optimised")
elapsed = time.time() - start
print(f"Optimised pipeline complete in {elapsed:.1f}s")
print(f"Rows written: {spark.table('perf_lab.data.results_optimised').count():,}")
print("\nCompare in the Spark UI: fewer Exchange nodes, less Shuffle Read/Write per stage.")

---

## Cleanup (optional)

Run the cell below to remove all lab resources from Unity Catalog once you have finished.

In [ ]:
# Uncomment and run to remove all lab resources
# spark.sql("DROP CATALOG IF EXISTS perf_lab CASCADE")
# print("✅ Lab resources cleaned up")